In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle
import cudf

In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

import pandas as pd

# load just the datasets q01 needs.
factor = 1
DATA_ROOT = f"/home/dias-benchmarks/tpch/data/factor_{factor}"
STORAGE_OPTS = {}  # or load from JSON

def load_lineitem(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = f"{data_folder}/lineitem.tbl"
    # parse the date columns on GPU during the read
    df = pd.read_csv(
        data_path,
        sep="|",
        parse_dates=["L_SHIPDATE", "L_RECEIPTDATE", "L_COMMITDATE"],
        **storage_options
    )
    print(df.columns)
    return df

lineitem = load_lineitem(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 1 ###

def load_orders(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = f"{data_folder}/orders.tbl"
    # parse O_ORDERDATE on GPU during CSV read to avoid a separate to_datetime pass
    df = pd.read_csv(
        data_path,
        sep='|',
        parse_dates=['O_ORDERDATE'],
        **storage_options
    )
    return df

orders = load_orders(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 2 ###

def load_customer(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/customer.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
customer = load_customer(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 3 ###

def load_nation(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/nation.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
nation = load_nation(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 4 ###

def load_region(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/region.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
region = load_region(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 5 ###

def load_supplier(
    data_folder: str, **storage_options
) -> pd.DataFrame:
    data_path = data_folder + "/supplier.tbl"
    # df = pd.read_parquet(
    #     data_path,  storage_options=storage_options
    # )
    df = pd.read_csv(data_path, sep="|")
    return df
supplier = load_supplier(DATA_ROOT, **STORAGE_OPTS)

In [ ]:
%%RecordEventWithColumnInfo
### cell 0 ###

# Filter on region and order year using GPU-friendly dt.year to avoid CPU-side Timestamp comparisons
rsel = region.R_NAME == "ASIA"
fregion = region[rsel]

# Use dt.year for a single GPU comparison instead of two CPU comparisons
osel = orders.O_ORDERDATE.dt.year == 1996
forders = orders[osel]

# Perform the sequence of merges as before (all GPU)
jn1 = fregion.merge(nation,   left_on="R_REGIONKEY", right_on="N_REGIONKEY")
jn2 = jn1.merge( customer,    left_on="N_NATIONKEY", right_on="C_NATIONKEY")
jn3 = jn2.merge( forders,     left_on="C_CUSTKEY",   right_on="O_CUSTKEY")
jn4 = jn3.merge( lineitem,    left_on="O_ORDERKEY",  right_on="L_ORDERKEY")
jn5 = supplier.merge(
    jn4,
    left_on=["S_SUPPKEY", "S_NATIONKEY"],
    right_on=["L_SUPPKEY", "N_NATIONKEY"]
)

# Compute revenue and aggregate
jn5["TMP"] = jn5.L_EXTENDEDPRICE * (1.0 - jn5.L_DISCOUNT)
total = (
    jn5.groupby("N_NAME", as_index=False, sort=False)["TMP"]
       .sum()
       .sort_values("TMP", ascending=False)
)